In [ ]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd
import shutil

import sys
import scipy.io as sio
import paste as pst
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.metrics import adjusted_rand_score

In [ ]:
def stack_obs(_dir):
    fds = os.listdir(_dir)
    _obs = []
    for fd in fds:
        if fd.startswith("R"):
            continue
        fns = os.listdir(join(_dir, fd))
        adx = sc.read_h5ad(join(_dir, fd, fns[0]))
        _obs.append(adx.obs)
    df = pd.concat(_obs)
    return df

In [ ]:
target_methods = [
    'CLUE',
    'StabMap',
    'SpaMosaic(non-spatial)',
    'UINMF',
    'scMoMaT',
    'SpaMosaic',
    'MIDAS',
    'Cobolt',
    'MultiVI'
]

data_dir = '../../data/processed/'
cluster_dir = '../../results/clusters/'
score_dir = '../../results/scores/'

### Lymph

In [ ]:
ds = 'Lymph-S1-S2-S3'

df_meta = stack_obs(join(data_dir, ds))
df_meta.head()

In [ ]:
labd = {
 'capsule':'Capsule',
 'cortex':'Cortex',
 'follicle':'Follicle',
 'hilum':'Hilum',
 'medulla Cords':'Medulla cords',
 'medulla cords':'Medulla cords',
 'medulla sinuses':'Medulla sinuses',
 'medulla vessel':'Medulla vessels',
 'medulla vessels':'Medulla vessels',
 'pericapsular adipose tissue':'Pericapsular adipose tissue',
 'subcapsular sinus':'Subcapsular sinus',
 'trabeculae':'Trabeculae'
}
df_meta['lab'] = df_meta['lab'].map(labd)

batch_key = 'src'
lab_key = 'lab'

In [ ]:
df_res = {}

for meth in target_methods:
    cluster_file = join(cluster_dir, meth, ds, 'df_cluster.csv')
    if not os.path.exists(cluster_file):
        print(f"[Skip] {cluster_file} not found.")
        continue

    df_cluster = pd.read_csv(cluster_file, index_col=0)

    common_idx = df_meta.index.intersection(df_cluster.index)
    if len(common_idx) == 0:
        print(f"[Warning] No overlapping cells between meta and cluster for {meth}")
        continue

    df_meta_aligned = df_meta.loc[common_idx].copy()
    df_cluster_aligned = df_cluster.loc[common_idx].copy()

    df_meta_aligned['cluster'] = df_cluster_aligned['cluster'].to_list()

    target_y = df_meta_aligned['lab'].to_list()
    predict_y = df_meta_aligned['cluster'].to_list()
    ari_all = adjusted_rand_score(target_y, predict_y)

    batch_aris = {}
    for batch_val in df_meta_aligned[batch_key].unique():
        subset = df_meta_aligned[df_meta_aligned[batch_key] == batch_val]
        if subset.shape[0] == 0:
            batch_aris[batch_val] = float('nan')
            continue
        batch_aris[batch_val] = adjusted_rand_score(subset['lab'], subset['cluster'])

    df_res[meth] = [ari_all] + [batch_aris[b] for b in sorted(batch_aris.keys())]

columns = ['all'] + sorted(df_meta[batch_key].unique())
df_res = pd.DataFrame.from_dict(df_res, orient='index', columns=columns)

os.makedirs(join(score_dir, ds), exist_ok=True)
df_res.to_csv(join(score_dir, ds, 'df_ARI.csv'))

### Tonsil-S1-S2-S3

In [ ]:
ds = 'Tonsil-S1-S2-S3'

df_meta = stack_obs(join(data_dir, ds))
df_meta.head()

In [ ]:
labd = {
    'connective & epithelial tissue': 'Connective and epithelial tissue',
    'connective and epithelial tissue': 'Connective and epithelial tissue',
    'germinal Center': 'Germinal Center',
    'germinal center': 'Germinal Center',
    'lymphoid follicle': 'Lymphoid follicle',
    'nan': 'Nan',
    'tonsillar parenchyma': 'Tonsillar parenchyma'
}

df_meta['lab'] = df_meta['lab'].map(labd)

batch_key = 'src'
lab_key = 'lab'

In [ ]:
df_res = {}

for meth in target_methods:
    cluster_file = join(cluster_dir, meth, ds, 'df_cluster.csv')
    if not os.path.exists(cluster_file):
        print(f"[Skip] {cluster_file} not found.")
        continue

    df_cluster = pd.read_csv(cluster_file, index_col=0)

    common_idx = df_meta.index.intersection(df_cluster.index)
    if len(common_idx) == 0:
        print(f"[Warning] No overlapping cells between meta and cluster for {meth}")
        continue

    df_meta_aligned = df_meta.loc[common_idx].copy()
    df_cluster_aligned = df_cluster.loc[common_idx].copy()

    df_meta_aligned['cluster'] = df_cluster_aligned['cluster'].to_list()

    target_y = df_meta_aligned['lab'].to_list()
    predict_y = df_meta_aligned['cluster'].to_list()
    ari_all = adjusted_rand_score(target_y, predict_y)

    batch_aris = {}
    for batch_val in df_meta_aligned[batch_key].unique():
        subset = df_meta_aligned[df_meta_aligned[batch_key] == batch_val]
        if subset.shape[0] == 0:
            batch_aris[batch_val] = float('nan')
            continue
        batch_aris[batch_val] = adjusted_rand_score(subset['lab'], subset['cluster'])

    df_res[meth] = [ari_all] + [batch_aris[b] for b in sorted(batch_aris.keys())]

columns = ['all'] + sorted(df_meta[batch_key].unique())
df_res = pd.DataFrame.from_dict(df_res, orient='index', columns=columns)

os.makedirs(join(score_dir, ds), exist_ok=True)
df_res.to_csv(join(score_dir, ds, 'df_ARI.csv'))

### Simulation

In [ ]:
def stack_obs2(data_dir):
    ad1 = sc.read_h5ad(join(data_dir, 'ad_rna1.h5ad'))
    ad2 = sc.read_h5ad(join(data_dir, 'ad_rna2.h5ad'))
    ad3 = sc.read_h5ad(join(data_dir, 'ad_adt2.h5ad'))
    
    ad1.obs['src'] = 's1'
    ad2.obs['src'] = 's2'
    ad3.obs['src'] = 's3'
    dft = pd.concat([ad1.obs, ad2.obs, ad3.obs])
    return dft

In [ ]:
ds = 'Simulations'

batch_key = 'src'
lab_key = 'gt'

folds = os.listdir(join(data_dir, ds))
for fd in folds:
    _dir = join(data_dir, fd)
    for seed in ['1', '2', '3']:
        df_meta = stack_obs2(join(data_dir, ds, fd, seed))
        
        df_res = {}
        for meth in target_methods:
            cluster_file = join(cluster_dir, meth, ds, fd, seed, 'df_cluster.csv')
            if not os.path.exists(cluster_file):
                print(f"[Skip] {cluster_file} not found.")
                continue

            df_cluster = pd.read_csv(cluster_file, index_col=0)

            common_idx = df_meta.index.intersection(df_cluster.index)
            if len(common_idx) == 0:
                print(f"[Warning] No overlapping cells between meta and cluster for {meth}")
                continue

            df_meta_aligned = df_meta.loc[common_idx].copy()
            df_cluster_aligned = df_cluster.loc[common_idx].copy()

            df_meta_aligned['cluster'] = df_cluster_aligned['cluster'].to_list()

            target_y = df_meta_aligned[lab_key].to_list()
            predict_y = df_meta_aligned['cluster'].to_list()
            ari_all = adjusted_rand_score(target_y, predict_y)

            batch_aris = {}
            for batch_val in df_meta_aligned[batch_key].unique():
                subset = df_meta_aligned[df_meta_aligned[batch_key] == batch_val]
                if subset.shape[0] == 0:
                    batch_aris[batch_val] = float('nan')
                    continue
                batch_aris[batch_val] = adjusted_rand_score(subset[lab_key], subset['cluster'])

            df_res[meth] = [ari_all] + [batch_aris[b] for b in sorted(batch_aris.keys())]

        columns = ['all'] + sorted(df_meta[batch_key].unique())
        df_res = pd.DataFrame.from_dict(df_res, orient='index', columns=columns)

        os.makedirs(join(score_dir, ds, fd, seed), exist_ok=True)
#         df_res.to_csv(join(score_dir, ds, fd, seed, 'df_ARI.csv'))